# 3.1 BPE 与 SentencePiece / BPE and SentencePiece

文本怎样变成编号。手写 BPE 的训练、编码和解码，再用真实的合并规则复现 GPT-2 的分词器。

How text becomes ids. Write BPE training, encoding and decoding by hand, then reproduce GPT-2's tokenizer from its real merge rules.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手写 BPE 的训练、编码与解码 / BPE training, encoding and decoding by hand

实现字节级 BPE 的五个函数。测试先用 aaabdaaabac 这个经典例子核对三次合并的结果，再在一段 856 字节的文本上用不同的词表大小训练，检查编码后能无损解码，并核对 token 数随词表增大而减少的具体数值。

Implement the five functions of byte-level BPE. The tests first check the three merges of the classic example aaabdaaabac, then train with several vocabulary sizes on a text of 856 bytes, check that encoding can be decoded without loss, and verify the exact token counts as the vocabulary grows.

In [ ]:
def pair_counts(ids):
    """Count every adjacent pair in the list `ids`.

    Returns a dict {(a, b): count}. Keys must be inserted in order of first appearance
    (a plain dict filled by one left-to-right pass does this), so that ties are broken
    the same way for everyone.
    """
    # TODO 1
    raise NotImplementedError


def merge(ids, pair, new_id):
    """Replace every non-overlapping occurrence of `pair` in `ids`, left to right, by `new_id`."""
    # TODO 2: walk through ids with an index; skip two items after a match
    raise NotImplementedError


def train_bpe(text, vocab_size):
    """Byte-level BPE training.

    Start from the UTF-8 bytes of `text` (ids 0..255). Repeatedly take the most frequent
    pair (first one on ties: max(counts, key=counts.get)), give it the next free id
    (256, 257, ...) and merge it, until the vocabulary has `vocab_size` entries or no pair is left.
    Returns the dict {(a, b): new_id} in the order the merges were learned.
    """
    # TODO 3
    raise NotImplementedError


def encode(text, merges):
    """Text -> list of token ids, applying the learned merges in the order they were learned."""
    ids = list(text.encode("utf-8"))
    # TODO 4: while a mergeable pair exists, merge the one that was learned EARLIEST
    #         (the pair with the smallest merges[pair])
    raise NotImplementedError


def decode(ids, merges):
    """List of token ids -> text."""
    # TODO 5: build {id: bytes} for 0..255 and for every merge, concatenate, decode as UTF-8
    #         (use errors="replace" so that a cut-off character does not crash)
    raise NotImplementedError

In [ ]:
# 检查 / checks
import codecs, contextlib, io

assert pair_counts([1, 2, 3, 1, 2]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1}
assert list(pair_counts([5, 6, 5, 6, 7])) == [(5, 6), (6, 5), (6, 7)], "keys in order of first appearance"
assert merge([5, 6, 6, 7, 9, 1], (6, 7), 99) == [5, 6, 99, 9, 1]
assert merge([1, 1, 1, 1, 1], (1, 1), 9) == [9, 9, 1], "non-overlapping, left to right"

# The classic example: aaabdaaabac
m = train_bpe("aaabdaaabac", 259)
assert m == {(97, 97): 256, (256, 97): 257, (257, 98): 258}, f"got {m}"
assert encode("aaabdaaabac", m) == [258, 100, 258, 97, 99], "11 bytes become 5 tokens"
assert decode([258, 100, 258, 97, 99], m) == "aaabdaaabac"
assert len(train_bpe("ab", 300)) == 1, "stop when no pair is left"

with contextlib.redirect_stdout(io.StringIO()):
    import this
text = codecs.decode(this.s, "rot13")                 # the Zen of Python, 856 bytes
lengths = {}
for V in (256, 300, 400, 600):
    merges = train_bpe(text, V)
    assert len(merges) == V - 256
    ids = encode(text, merges)
    assert decode(ids, merges) == text, "encode then decode must give the text back"
    lengths[V] = len(ids)
assert lengths == {256: 856, 300: 499, 400: 292, 600: 92}, f"got {lengths}"

merges = train_bpe(text, 300)
assert list(merges.items())[:3] == [((115, 32), 256), ((32, 116), 257), ((32, 105), 258)], "first merges: 's ', ' t', ' i'"
# Text the tokenizer has never seen still works, because every byte is in the vocabulary.
new = "better than ugly 北京"
ids = encode(new, merges)
assert ids == [276, 32, 117, 103, 108, 269, 229, 140, 151, 228, 186, 172], f"got {ids}"
assert decode(ids, merges) == new
assert decode([229, 140], merges) == chr(0xFFFD), "half a Chinese character is not valid text"
print("m31_bpe: all tests passed")
print("  tokens for the 856-byte text by vocabulary size:", lengths)
print("  pieces:", [decode([i], merges) for i in encode("Simple is better than complex.", merges)])

<details><summary><b>提示 1 / Hint 1</b></summary>

统计相邻对用 `zip(ids, ids[1:])`。普通的字典会保持键的插入顺序，正好满足“先出现的优先”的要求。

Count adjacent pairs with `zip(ids, ids[1:])`. A plain dict keeps the insertion order of keys, which gives the first-appearance tie-breaking required here.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`merge` 用下标逐个扫描。当前位置和下一个位置正好是要合并的那一对时，写入新编号并跳过两个位置，否则原样写入并前进一个位置。

`merge` scans with an index. When the current and the next item are the pair to merge, append the new id and skip two items. Otherwise append the item and advance by one.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

编码时每一轮要找的是当前序列里“最早学到”的那条合并，即 `min(counts, key=lambda p: merges.get(p, float('inf')))`。如果找到的这一对不在 `merges` 里，说明已经没有可以合并的了。

Each round of encoding looks for the merge that was learned earliest among the pairs present, `min(counts, key=lambda p: merges.get(p, float('inf')))`. If that pair is not in `merges`, nothing more can be merged.

</details>

<details><summary><b>提示 4 / Hint 4</b></summary>

解码时先建立 `{编号: 字节串}` 的表。按 `merges` 的顺序遍历，每条合并的字节串是两个组成部分的字节串相加。

For decoding first build a table `{id: bytes}`. Walk through `merges` in order, where the bytes of each merge are the concatenation of the bytes of its two parts.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
def pair_counts(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def train_bpe(text, vocab_size):
    ids = list(text.encode("utf-8"))
    merges = {}
    for new_id in range(256, vocab_size):
        counts = pair_counts(ids)
        if not counts:
            break
        pair = max(counts, key=counts.get)        # most frequent pair, first one on ties
        merges[pair] = new_id
        ids = merge(ids, pair, new_id)
    return merges


def encode(text, merges):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        counts = pair_counts(ids)
        pair = min(counts, key=lambda p: merges.get(p, float("inf")))   # earliest learned merge present
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids


def decode(ids, merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for (a, b), new_id in merges.items():          # dicts keep insertion order, so a and b already exist
        vocab[new_id] = vocab[a] + vocab[b]
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")
```

</details>

## 2. 用真实的合并规则复现 GPT-2 分词器 / Reproduce GPT-2's tokenizer from its real merge rules

GPT-2 的五万条合并规则保存在 `merges.txt` 中。实现对单个预分词单位应用这些规则的函数，再接上预分词，得到完整的分词结果。测试会在英文、数字、中文和代码上与 Hugging Face 的分词器逐 token 对比，并核对编号与 tiktoken 一致。

The fifty thousand merge rules of GPT-2 are stored in `merges.txt`. Implement the function that applies them to a single pre-token, then combine it with pre-tokenization to obtain the full tokenization. The tests compare token by token with the Hugging Face tokenizer on English, digits, Chinese and code, and check that the ids agree with tiktoken.

需要联网下载模型。/ Downloads a model.

In [ ]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer


def load_gpt2_ranks():
    """GPT-2's 50,000 learned merges, in the order they were learned. Nothing to change.

    Returns {(left, right): rank}. Rank 0 was learned first. In this file a leading space
    is written as the character 'Ġ', and other bytes are mapped to printable characters.
    """
    path = hf_hub_download("openai-community/gpt2", "merges.txt")
    lines = open(path, encoding="utf-8").read().split("\n")[1:]
    return {tuple(line.split()): rank for rank, line in enumerate(lines) if line}


def bpe_word(word, ranks):
    """Apply the merges to ONE pre-token, e.g. 'Ġstrawberry'. Returns a list of strings.

    Start from the single characters. While some adjacent pair is in `ranks`, take the pair
    with the LOWEST rank and merge every occurrence of it.
    """
    symbols = list(word)
    # TODO 1
    raise NotImplementedError


def gpt2_tokenize(text, tok, ranks):
    """Tokenize a whole text like GPT-2 does. Returns a list of token strings.

    Step 1 (given by the library): split the text into pre-tokens with
        tok.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)   -> [(pre_token, span), ...]
    Step 2 (yours): run bpe_word on every pre-token and concatenate the results.
    """
    # TODO 2
    raise NotImplementedError

In [ ]:
# 检查 / checks
import tiktoken
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
ranks = load_gpt2_ranks()
assert len(ranks) == 50000 and ranks[("Ġ", "t")] == 0, "the first merge GPT-2 learned is space + t"

assert bpe_word("Ġstrawberry", ranks) == ["Ġstrawberry"], "with a leading space this common word is one token"
assert bpe_word("strawberry", ranks) == ["st", "raw", "berry"], "without the space it falls apart into three"
assert bpe_word("1234567", ranks) == ["123", "45", "67"]

texts = ["The capital of France is Paris.",
         "Tokenization isn't obvious: strawberry vs. strawberry!",
         "1234567 + 7654321 = 8888888",
         "北京是中国的首都。München liegt in Bayern.",
         "def f(x):\n    return x ** 2  # square"]
enc = tiktoken.get_encoding("gpt2")
for t in texts:
    mine = gpt2_tokenize(t, tok, ranks)
    assert mine == tok.tokenize(t), f"differs from the Hugging Face tokenizer on {t!r}"
    assert tok.convert_tokens_to_ids(mine) == enc.encode(t), f"token ids differ from tiktoken on {t!r}"
print("m31_gpt2_bpe: all tests passed. Your code reproduces GPT-2's tokenizer.")
print(" ", gpt2_tokenize("Tokenization isn't obvious", tok, ranks))

<details><summary><b>提示 1 / Hint 1</b></summary>

每一轮先收集当前所有的相邻对 `set(zip(symbols, symbols[1:]))`，从中选出 `ranks` 值最小的一对。

Each round first collects all adjacent pairs, `set(zip(symbols, symbols[1:]))`, and picks the one with the smallest value in `ranks`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

选出的这一对要把所有出现的位置都合并，写法与上一题的 `merge` 相同，只是这里合并的结果是两个字符串相加。

Every occurrence of the chosen pair is merged, written like `merge` in the previous exercise, except that here the merged result is the concatenation of two strings.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`gpt2_tokenize` 只需对预分词返回的每个 `(片段, 位置)` 取第一个元素调用 `bpe_word`，再把结果依次接起来。

`gpt2_tokenize` only has to call `bpe_word` on the first element of every `(piece, span)` returned by the pre-tokenizer and concatenate the results.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer


def load_gpt2_ranks():
    path = hf_hub_download("openai-community/gpt2", "merges.txt")
    lines = open(path, encoding="utf-8").read().split("\n")[1:]
    return {tuple(line.split()): rank for rank, line in enumerate(lines) if line}


def bpe_word(word, ranks):
    symbols = list(word)
    while len(symbols) >= 2:
        pairs = set(zip(symbols, symbols[1:]))
        best = min(pairs, key=lambda p: ranks.get(p, float("inf")))
        if best not in ranks:
            break
        merged, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best:
                merged.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        symbols = merged
    return symbols


def gpt2_tokenize(text, tok, ranks):
    out = []
    for pre_token, _ in tok.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text):
        out.extend(bpe_word(pre_token, ranks))
    return out
```

</details>

## 3. 用 SentencePiece 训练并观察 / Train and inspect with SentencePiece

用 SentencePiece 在一小段文本上分别训练 BPE 和 unigram 两种模型，观察空格怎样被写成 `▁`，并手写一个不依赖库的还原函数。测试还会在没有空格的中文文本上训练，确认不需要预先分词。

Train a BPE and a unigram model with SentencePiece on a small text, observe how spaces are written as `▁`, and write a detokenizer that does not use the library. The tests also train on Chinese text without spaces to confirm that no prior word segmentation is needed.

In [ ]:
import io
import sentencepiece as spm


def train_sentencepiece(lines, vocab_size, model_type="bpe"):
    """Train a SentencePiece model in memory and return a ready SentencePieceProcessor.

    lines: list of strings (raw text, no pre-tokenization needed).  model_type: "bpe" or "unigram".
    """
    model = io.BytesIO()
    # TODO 1: spm.SentencePieceTrainer.train(sentence_iterator=iter(lines), model_writer=model,
    #                                         vocab_size=..., model_type=..., minloglevel=2)
    # TODO 2: return spm.SentencePieceProcessor(model_proto=model.getvalue())
    raise NotImplementedError


def detokenize(pieces):
    """Turn SentencePiece pieces back into text WITHOUT the library.

    SentencePiece writes a space as the visible symbol '▁' (U+2581) at the start of a piece.
    """
    # TODO 3: join the pieces, turn '▁' back into a space, drop the space at the very beginning
    raise NotImplementedError

In [ ]:
# 检查 / checks
import codecs, contextlib, io

with contextlib.redirect_stdout(io.StringIO()):
    import this
lines = [l for l in codecs.decode(this.s, "rot13").split("\n") if l]

bpe = train_sentencepiece(lines, 120, "bpe")
uni = train_sentencepiece(lines, 120, "unigram")
assert bpe.get_piece_size() == 120 and uni.get_piece_size() == 120

s = "Simple is better than complex."
pb, pu = bpe.encode(s, out_type=str), uni.encode(s, out_type=str)
assert pb[0].startswith("▁"), "the first piece carries the word-boundary marker"
assert sum(p.startswith("▁") for p in pb) == 5, "five words, five pieces that start with the marker"
assert pb != pu, "BPE and unigram segment the same sentence differently"
assert detokenize(pb) == s and detokenize(pu) == s, "joining the pieces must give the text back"
assert bpe.decode(bpe.encode(s)) == s

# No language-specific pre-tokenization: text without spaces works the same way.
zh = ["今天天气很好", "今天我们学习分词", "分词很重要", "天气很重要"] * 5
z = train_sentencepiece(zh, 20, "bpe")
pz = z.encode("今天分词", out_type=str)
assert detokenize(pz) == "今天分词"
assert len(pz) < len("今天分词") + 1, "frequent character pairs should have been merged"
print("m31_sp: all tests passed")
print("  bpe    :", pb)
print("  unigram:", pu)
print("  zh     :", pz)

<details><summary><b>提示 1 / Hint 1</b></summary>

训练结果写进内存里的 `io.BytesIO()`，再用 `model_proto=model.getvalue()` 创建处理器，不需要落盘。

The trained model is written to an in-memory `io.BytesIO()` and the processor is created with `model_proto=model.getvalue()`, without touching the disk.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

还原时把所有片段直接拼接，把 `▁` 换成空格，再去掉最开头多出来的那个空格。

To detokenize, join all pieces, replace `▁` by a space and remove the extra space at the very beginning.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import io
import sentencepiece as spm


def train_sentencepiece(lines, vocab_size, model_type="bpe"):
    model = io.BytesIO()
    spm.SentencePieceTrainer.train(sentence_iterator=iter(lines), model_writer=model,
                                   vocab_size=vocab_size, model_type=model_type, minloglevel=2)
    return spm.SentencePieceProcessor(model_proto=model.getvalue())


def detokenize(pieces):
    text = "".join(pieces).replace("▁", " ")
    return text[1:] if text.startswith(" ") else text
```

</details>

## 4. 词表大小的取舍与手算 BPE / The vocabulary trade-off and BPE by hand

在纸上完成。
(a) 对字符串 `abababcabc` 手动做两次 BPE 合并，写出每次合并的是哪一对，以及合并后的序列。出现次数相同时取先出现的一对。
(b) 向量宽度 $d = 4096$。词表从 5 万增加到 20 万，嵌入表增加多少参数？
(c) 某段文本用字符级分词是 4000 个 token，用子词分词是 1000 个 token。注意力部分的计算量相差多少倍？
(d) 用一两句话分别说明按字符切分和按单词切分的主要问题。

Work on paper.
(a) Do two BPE merges by hand on the string `abababcabc`. State which pair is merged each time and the sequence afterwards. On equal counts take the pair that appears first.
(b) The vector width is $d = 4096$. How many parameters does the embedding table gain when the vocabulary grows from 50 thousand to 200 thousand?
(c) A text is 4000 tokens with character-level and 1000 tokens with subword tokenization. By what factor does the computation of the attention part differ?
(d) State in a sentence or two each the main problem of splitting by characters and of splitting by words.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 先数出 ab、ba、bc、ca 各出现几次。

(a) First count how often ab, ba, bc and ca occur.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 注意力部分的计算量与序列长度的平方成正比。

(c) The computation of the attention part is proportional to the square of the sequence length.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 初始序列是 a b a b a b c a b c。ab 出现 4 次，ba 出现 2 次，bc 出现 2 次，ca 出现 1 次。第一次合并 ab，记作 X，得到 X X X c X c。此时 XX 出现 2 次，Xc 出现 2 次，cX 出现 1 次，先出现的是 XX，记作 Y，得到 Y X c X c。

(b) 增加 $150000 \times 4096 = 6.144 \times 10^{8}$，约 6.1 亿个参数。

(c) 序列长度相差 4 倍，注意力部分的计算量相差 16 倍。

(d) 按字符切分时序列太长，计算代价高，模型还要自己把字符组合成有意义的单位。按单词切分时词表无法穷尽所有词形和新词，遇到词表之外的词无法表示。

---

(a) The initial sequence is a b a b a b c a b c. ab occurs 4 times, ba twice, bc twice and ca once. The first merge is ab, call it X, giving X X X c X c. Now XX occurs twice, Xc twice and cX once. XX appears first, call it Y, giving Y X c X c.

(b) The gain is $150000 \times 4096 = 6.144 \times 10^{8}$, about 610 million parameters.

(c) The sequence lengths differ by a factor of 4, so the computation of the attention part differs by a factor of 16.

(d) With characters the sequences are too long, computation is expensive and the model has to assemble characters into meaningful units itself. With words the vocabulary cannot cover all word forms and new words, and a word outside it cannot be represented.

</details>